## Best practices
1. Preprocessing and cleaning
2. Train Test Split
3. BOW, TFIDF, Word2Vec
4. Train ML algorithm

In [46]:
## load the dataset
import pandas as pd
data=pd.read_csv('all_kindle_review.csv')
data.head()

,asin,helpful,rating,reviewText,reviewTime,reviewerID,reviewerName,summary,unixReviewTime
0,B00000001,2/2,2,This Kindle book review sample number 1. Inter...,"02 02, 2014",A00000000000001,Reviewer1,Review summary 1,1400086400
1,B00000002,3/3,3,This Kindle book review sample number 2. Inter...,"03 03, 2014",A00000000000002,Reviewer2,Review summary 2,1400172800
2,B00000003,4/4,4,This Kindle book review sample number 3. Inter...,"04 04, 2014",A00000000000003,Reviewer3,Review summary 3,1400259200
3,B00000004,5/5,5,This Kindle book review sample number 4. Inter...,"05 05, 2014",A00000000000004,Reviewer4,Review summary 4,1400345600
4,B00000005,1/1,1,This Kindle book review sample number 5. Inter...,"06 06, 2014",A00000000000005,Reviewer5,Review summary 5,1400432000


In [47]:
df=data[['reviewText','rating']]
df.head()

,reviewText,rating
0,This Kindle book review sample number 1. Inter...,2
1,This Kindle book review sample number 2. Inter...,3
2,This Kindle book review sample number 3. Inter...,4
3,This Kindle book review sample number 4. Inter...,5
4,This Kindle book review sample number 5. Inter...,1


In [48]:
df.shape

(120, 2)

In [49]:
## missinf Values
df.isnull().sum()

reviewText    0
rating        0
dtype: int64

In [50]:
df['rating'].unique()

array([2, 3, 4, 5, 1])

In [51]:
df['rating'].value_counts()

rating
2    24
3    24
4    24
5    24
1    24
Name: count, dtype: int64

In [52]:
## preprocessing and cleaning
## positive review is 1 and negative review is 0
df['rating'] = df['rating'].apply(lambda x:0 if x<3 else 1)

In [53]:
df.head()

,reviewText,rating
0,This Kindle book review sample number 1. Inter...,0
1,This Kindle book review sample number 2. Inter...,1
2,This Kindle book review sample number 3. Inter...,1
3,This Kindle book review sample number 4. Inter...,1
4,This Kindle book review sample number 5. Inter...,0


In [54]:
df['rating'].value_counts()

rating
1    72
0    48
Name: count, dtype: int64

In [55]:
## 1. Lower all the cases
df['reviewText'] = df['reviewText'].str.lower()

In [56]:
df.head()

,reviewText,rating
0,this kindle book review sample number 1. inter...,0
1,this kindle book review sample number 2. inter...,1
2,this kindle book review sample number 3. inter...,1
3,this kindle book review sample number 4. inter...,1
4,this kindle book review sample number 5. inter...,0


In [42]:
!pip install beautifulsoup4 lxml

Defaulting to user installation because normal site-packages is not writeable
  Obtaining dependency information for beautifulsoup4 from https://files.pythonhosted.org/packages/88/c6/92fcd42f1ba33e1184263f25bfabf3d27c383410470f169e4b8163bf9c17/beautifulsoup4-4.15.0-py3-none-any.whl.metadata
  Obtaining dependency information for lxml from https://files.pythonhosted.org/packages/61/9d/2e2f7d876349f45e0f3e29f72da311668853d59b58d473a2dea4f0160135/lxml-6.1.1-cp311-cp311-win_amd64.whl.metadata
  Obtaining dependency information for soupsieve>=1.6.1 from https://files.pythonhosted.org/packages/5e/f5/0c41cb68dcae6b7de4fac4188a3a9589e21fb31df21ea3a2e888db95e6c9/soupsieve-2.8.4-py3-none-any.whl.metadata
   ---------------------------------------- 0.0/109.9 kB ? eta -:--:--
   ---------------------------------------- 0.0/109.9 kB ? eta -:--:--
   --- ------------------------------------ 10.2/109.9 kB ? eta -:--:--
   ---------- ---------------------------- 30.7/109.9 kB 435.7 kB/s eta 0:00:01
  


[notice] A new release of pip is available: 23.2.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [43]:
import re
from nltk.corpus import stopwords
from bs4 import BeautifulSoup

stop_words = set(stopwords.words('english'))

In [57]:
## removing special character
df['reviewText'] = df['reviewText'].apply(lambda x: re.sub('[^a-z A-Z 0-9]+', '', x))
## remove the stopwords
df['reviewText'] = df['reviewText'].apply(lambda x: " ".join([y for y in x.split() if y not in stop_words]))
## remove url
df['reviewText'] = df['reviewText'].apply(lambda x: re.sub(r'http\S+|www\S+|ftp\S+', '', x))
## remove html tag
df['reviewText'] = df['reviewText'].apply(lambda x: BeautifulSoup(x, 'lxml').get_text())
## remove any additional spaces
df['reviewText'] = df['reviewText'].apply(lambda x: " ".join(x.split()))


In [58]:
df.head()

,reviewText,rating
0,kindle book review sample number 1 interesting...,0
1,kindle book review sample number 2 interesting...,1
2,kindle book review sample number 3 interesting...,1
3,kindle book review sample number 4 interesting...,1
4,kindle book review sample number 5 interesting...,0


In [61]:
## lemmatizer
import nltk
from nltk.stem import WordNetLemmatizer

In [62]:
lemmatizer=WordNetLemmatizer()

In [63]:
def lemmatize_words(text):
    return " ".join([lemmatizer.lemmatize(word) for word in text.split()])

In [64]:
df['reviewText'] = df['reviewText'].apply(lambda x:lemmatize_words(x))

In [65]:
df.head()

,reviewText,rating
0,kindle book review sample number 1 interesting...,0
1,kindle book review sample number 2 interesting...,1
2,kindle book review sample number 3 interesting...,1
3,kindle book review sample number 4 interesting...,1
4,kindle book review sample number 5 interesting...,0


In [68]:
## train test split
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(df['reviewText'], df['rating'], test_size=0.20)

In [73]:
from sklearn.feature_extraction.text import CountVectorizer
bow=CountVectorizer()
x_train_bow=bow.fit_transform(x_train).toarray()
x_test_bow=bow.transform(x_test).toarray()

In [77]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf=TfidfVectorizer()
x_train_tfidf=tfidf.fit_transform(x_train).toarray()
x_test_tfidf=tfidf.transform(x_test).toarray()

In [78]:
x_train_bow

array([[0, 0, 0, ..., 1, 1, 1],
       [0, 0, 0, ..., 1, 1, 1],
       [0, 0, 0, ..., 1, 1, 1],
       ...,
       [0, 0, 0, ..., 1, 1, 1],
       [0, 0, 0, ..., 1, 1, 1],
       [0, 0, 0, ..., 1, 1, 1]], shape=(96, 96))

In [79]:
from sklearn.naive_bayes import GaussianNB
nb_model_bow=GaussianNB().fit(x_train_bow,y_train)
nb_model_tfidf=GaussianNB().fit(x_train_tfidf,y_train)


In [81]:
from sklearn.metrics import confusion_matrix,accuracy_score,classification_report

In [82]:
y_pred_bow=nb_model_bow.predict(x_test_bow)

In [83]:
y_pred_tfidf=nb_model_tfidf.predict(x_test_tfidf)


In [85]:
confusion_matrix(y_test,y_pred_bow)

array([[13,  0],
       [11,  0]])

In [84]:
print("BOW accuracy: ",accuracy_score(y_test,y_pred_bow))

BOW accuracy:  0.5416666666666666


In [86]:
print("TFIDF accuracy: ",accuracy_score(y_test,y_pred_tfidf))


TFIDF accuracy:  0.5416666666666666
